# Leadgenerator — kansrijke adressen
**Takkenkamp Vastgoed Verduurzamers**

## Waar gaat dit notebook over?
Dit notebook zoekt adressen die een grote kans hebben om klant te worden bij Takkenkamp. Het werkt zo: we kijken naar adressen die al klant zijn (uit TIOS), het model leert welke kenmerken die adressen gemeen hebben (bouwjaar, oppervlakte, buurt, enzovoort), en daarna krijgt elk ander adres een kans-score. Het eindresultaat is een Excel-lijst met adressen, van meest naar minst kansrijk.

## De stappen in het kort
1. **Instellingen** – een paar vaste waarden die je zelf kunt aanpassen.
2. **Bestanden uploaden** – TIOS en Assetmaps inlezen.
3. **Data klaarzetten** – kolomnamen opschonen, TIOS samenvatten per adres, CBS-cijfers (gasverbruik en inkomen) ophalen.
4. **Koppelen** – TIOS en Assetmaps samenvoegen en bepalen wie al klant is.
5. **Opschonen** – ontbrekende waarden netjes afhandelen.
6. **Nieuwe kenmerken** – bijvoorbeeld "gebouwd vóór 1975".
7. **Buurteffect** – hoeveel klanten wonen er al in de buurt?
8. **Kenmerken kiezen** – welke kolommen het model mag gebruiken.
9. **Voorbewerking** – de data in een vorm zetten die het model snapt.
10. **Model testen** – hoe goed is het model? (AUC en precision@k)
11. **Wat drijft het model?** – welke kenmerken het zwaarst wegen.
12. **Leadlijst maken** – alle niet-klanten scoren en naar Excel schrijven.

## Welk model gebruiken we?
Eén vast model: een **ensemble** van *logistic regression* en *RandomForest*. Beide modellen geven per adres een kans, en het notebook neemt daar het gemiddelde van. In eerdere tests was deze combinatie beter dan elk los model. Dit is precies het model dat de leadlijst in stap 12 maakt.

## Wat is er in deze versie veranderd?
- Er wordt geen modelvergelijking meer gedaan (XGBoost, LightGBM, HistGradientBoosting en de losse modellen zijn eruit). Alleen het ensemble wordt nog getraind.
- Kruisvalidatie met **3 folds** in plaats van 5.
- AUC en precision@k worden in **één** ronde berekend. Voorheen werd het model hiervoor twee keer apart getraind.
- De twee modellen worden op één plek gedefinieerd (functie `maak_ensemble` in stap 10), zodat testen en de leadlijst gegarandeerd hetzelfde model gebruiken.
- Bij elk codeblok staat nu een eenvoudige uitleg van wat het doet.
- `gemeente_klanten_nabij` is verwijderd. Dit kenmerk maakte het model slechter (negatieve feature-belangrijkheid).
- Het buurteffect wordt voor de trainingsdata nu in stukken berekend in plaats van "totaal min het adres zelf". Die oude manier liet het model een trucje leren dat bij nieuwe adressen niet werkt (zie stap 7).

## ⚠️ Let op vóór je begint
De zichtbare kolomkoppen **"BAG-ID"** en **"Pand-ID"/"VBO-ID"** staan in de twee bronnen **verwisseld**. Dit notebook koppelt daarom niet op de zichtbare koptekst, maar op de technische naam tussen haakjes (`vbo_id`, met `pand_id` als terugval). Controleer bij oplevering zelf een paar rijen met de hand.


## 1. Instellingen

**Wat doet dit blok?**
Hier staan een paar vaste waarden die de rest van het notebook gebruikt. Zo hoef je ze maar op één plek aan te passen.

- `TIOS_BLAD` en `ASSETMAPS_BLAD`: de naam van het tabblad in Excel waar de data staat. Weet je de naam niet zeker? Laat het staan. Klopt de naam niet, dan pakt het notebook vanzelf het eerste tabblad en laat het zien welke tabbladen er wél zijn.
- `MISSING_CODE`: Assetmaps gebruikt het getal -99997 als het iets niet weet. Dat is geen echte waarde, dus we moeten het later als "onbekend" behandelen.
- `HUIDIG_JAAR`: nodig om uit te rekenen hoe oud een pand is.
- `ORDER_STATUSSEN`: welke statussen in TIOS betekenen dat iemand echt klant is geworden. Een adres met één van deze statussen telt als klant.


In [ ]:
TIOS_BLAD = "WoningBron"
ASSETMAPS_BLAD = "Blad1"        # pas aan naar jouw bladnaam, of laat staan (zie hierboven)

MISSING_CODE = -99997            # code voor "onbekend" in Assetmaps, geen echte 0-waarde
HUIDIG_JAAR = 2026                # voor de pandleeftijd (stap 6)

# Statussen in TIOS die een order betekenen. Controleer dit tegen de echte
# waarden in de kolom "status" voordat je het model traint. Getest tegen de
# aangeleverde data: "Opdracht" en "Opdracht o.v.v. Subsidie" bleken naast
# "Overgedragen aan Planning" ook afgeronde orders te zijn.
ORDER_STATUSSEN = [
    "Overgedragen aan Planning",
    "Opdracht",
    "Opdracht o.v.v. Subsidie",
]


# Technische kolomnamen die dit notebook in TIOS verwacht (het stukje tussen
# haakjes in de Excel-kopregel, bijv. "Aanmaakdatum (datum)" -> "datum").
# Klopt een naam niet met jouw export? Pas hem hier aan. Het notebook meldt
# straks duidelijk welke kolommen er wél zijn als er iets niet gevonden wordt,
# in plaats van een kale KeyError te geven.
TIOS_KOLOM_DATUM = "datum"
TIOS_KOLOM_STATUS = "status"
TIOS_KOLOM_ID = "id"
TIOS_KOLOM_OFFERTES = "offertes"
TIOS_KOLOM_SOORT = "soort"


## 2. Bestanden uploaden

**Wat doet dit blok?**
Dit blok zorgt dat het notebook je Excel-bestanden kan vinden.

- **In Google Colab** krijg je een uploadknop. Je mag 1 bestand kiezen (met beide tabbladen) of 2 losse bestanden (TIOS + Assetmaps). Het notebook herkent het Assetmaps-bestand aan het woord "asset" in de bestandsnaam, dus zet dat erin.
- **Lokaal op je eigen computer** is er geen uploadknop. Dan vul je onderaan zelf de bestandsnamen in.

Aan het eind print het welke bestanden het gaat gebruiken, zodat je kunt controleren of dat klopt.


In [2]:
try:
    import google.colab
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "openpyxl"], check=False)

    print("Upload je bestand(en): 1 bestand met beide bladen, of 2 losse bestanden.")
    uploaded = files.upload()
    bestanden = list(uploaded.keys())

    if len(bestanden) == 1:
        TIOS_BESTAND = ASSETMAPS_BESTAND = bestanden[0]
    else:
        assetmaps_kandidaten = [b for b in bestanden if "asset" in b.lower()]
        ASSETMAPS_BESTAND = assetmaps_kandidaten[0] if assetmaps_kandidaten else bestanden[1]
        TIOS_BESTAND = next(b for b in bestanden if b != ASSETMAPS_BESTAND)
else:
    # Draai je dit lokaal (niet in Colab)? Zet hier zelf de bestandsnamen/paden.
    TIOS_BESTAND = "Adressenbestand_2016_2023.xlsx"
    ASSETMAPS_BESTAND = "Woningen_Zeeland.xlsx"

print("TIOS-bestand:      ", TIOS_BESTAND)
print("Assetmaps-bestand: ", ASSETMAPS_BESTAND)


TIOS-bestand:       Adressenbestand_2016_2023.xlsx
Assetmaps-bestand:  Woningen_Zeeland.xlsx


### Benodigde bibliotheken laden

**Wat doet dit blok?**
Python heeft voor veel taken kant-en-klare "gereedschapskisten" (bibliotheken). Hier laden we de kisten die we nodig hebben:

- `pandas` en `numpy`: werken met tabellen en getallen.
- `requests`: gegevens ophalen van internet (de CBS-cijfers).
- `scikit-learn` (`sklearn`): het machine-learninggedeelte. De modellen (logistic regression en RandomForest), het opvullen van lege waarden, het testen met kruisvalidatie en het meten van de kwaliteit.

XGBoost en LightGBM worden niet meer geladen, want die modellen gebruiken we niet meer.


In [ ]:
import re
import time
import warnings

import numpy as np
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Waarschuwingen verbergen, zodat de uitvoer overzichtelijk blijft
warnings.filterwarnings("ignore")


## 3. Bestanden inlezen en kolomnamen opschonen

**Wat doet dit blok?**
Dit blok leest de twee Excel-bestanden in en maakt de kolomnamen korter en eenduidiger.

De kolomkoppen in beide bestanden zien er zo uit: `"Zichtbare naam (technische_naam)"`. Het stukje tussen haakjes is de echte, technische naam. De functie `technische_namen` pakt alleen dat stukje. Zo heet de kolom daarna bijvoorbeeld gewoon `vbo_id`. Dat voorkomt de verwarring rond BAG-ID/Pand-ID/VBO-ID.

De functie `lees_blad` leest het juiste tabblad. Bestaat dat tabblad niet, dan pakt hij het eerste en meldt hij dat.

Aan het eind zie je hoeveel rijen er zijn ingelezen, hoe lang dat duurde en welke kolommen er nu zijn.


In [ ]:
def technische_namen(df):
    """Haalt de naam tussen haakjes uit elke kolomkop en gebruikt die."""
    nieuwe_namen = {}
    for kolom in df.columns:
        match = re.search(r"\(([^)]+)\)\s*$", str(kolom).strip())
        nieuwe_namen[kolom] = match.group(1) if match else kolom
    return df.rename(columns=nieuwe_namen)


def lees_blad(bestand, gewenst_blad):
    """Leest het gevraagde blad. Bestaat dat blad niet (verkeerde naam
    of typefout), dan pakt deze functie automatisch het eerste blad en
    laat zien welke bladen er echt in het bestand zitten, zodat je de
    naam hierboven bij Instellingen makkelijk kunt corrigeren."""
    beschikbare_bladen = pd.ExcelFile(bestand).sheet_names
    if gewenst_blad in beschikbare_bladen:
        return pd.read_excel(bestand, sheet_name=gewenst_blad)

    eerste_blad = beschikbare_bladen[0]
    print(
        f"Let op: blad '{gewenst_blad}' niet gevonden in '{bestand}'. "
        f"Beschikbare bladen: {beschikbare_bladen}. "
        f"Gebruik nu automatisch het eerste blad: '{eerste_blad}'."
    )
    return pd.read_excel(bestand, sheet_name=eerste_blad)


_t0 = time.time()
tios = lees_blad(TIOS_BESTAND, TIOS_BLAD)
tios = technische_namen(tios)
print(f"TIOS ingelezen: {len(tios)} rijen in {time.time() - _t0:.1f} seconden.")

_t0 = time.time()
assetmaps = lees_blad(ASSETMAPS_BESTAND, ASSETMAPS_BLAD)
assetmaps = technische_namen(assetmaps)
print(f"Assetmaps ingelezen: {len(assetmaps)} rijen in {time.time() - _t0:.1f} seconden.")

print("TIOS-kolommen na omzetten:     ", list(tios.columns))
print("Assetmaps-kolommen na omzetten:", list(assetmaps.columns))


# Check: staan de kolommen waar het notebook straks vanuit gaat er echt in?
# Zo niet, dan meld je dat nu al duidelijk, in plaats van verderop een kale
# KeyError te krijgen. Komt de technische naam tussen haakjes in jouw export
# niet overeen met de instelling? Pas de bijbehorende TIOS_KOLOM_...-instelling
# in stap 1 aan naar de naam die hieronder wél verschijnt.
_verwachte_tios_kolommen = {
    "DATUM": TIOS_KOLOM_DATUM, "STATUS": TIOS_KOLOM_STATUS, "ID": TIOS_KOLOM_ID,
    "OFFERTES": TIOS_KOLOM_OFFERTES, "SOORT": TIOS_KOLOM_SOORT,
    "VBO_ID": "vbo_id", "PAND_ID": "pand_id",
}
_ontbrekend_tios = {label: naam for label, naam in _verwachte_tios_kolommen.items() if naam not in tios.columns}
if _ontbrekend_tios:
    print("\nLet op: onderstaande kolommen zijn NIET gevonden in TIOS na het omzetten "
          "van de kolomnamen:")
    for label, naam in _ontbrekend_tios.items():
        print(f"  - verwacht als '{naam}' (instelling TIOS_KOLOM_{label}) — niet gevonden")
    print("Pas de bijbehorende instelling in stap 1 aan naar de juiste technische naam "
          "(het stukje tussen haakjes in de kopregel van jouw Excel-bestand).")
    print(f"Beschikbare kolommen in TIOS: {list(tios.columns)}")
else:
    print("\nAlle verwachte TIOS-kolommen zijn gevonden.")


### Extra check: wat betekenen de kolommen `huur`, `koop` en `woz` precies?

Voordat we kiezen welke van deze drie we gebruiken, eerst even kijken wat erin zit: zijn het percentages (bijv. "70% koopwoningen in de buurt"), of geschatte geldbedragen per pand (zoals de WOZ-waarde)? Dat bepaalt of "alleen koopwoningen" betekent "gebruik de kolom `koop`", of dat we eigenlijk een heel andere kolom nodig hebben.


In [ ]:
for kolom in ["huur", "koop", "woz"]:
    if kolom in assetmaps.columns:
        reeks = assetmaps[kolom].replace(MISSING_CODE, np.nan)
        print(f"{kolom}: min={reeks.min()}, max={reeks.max()}, gemiddeld={reeks.mean():.1f}, "
              f"aantal unieke waarden={reeks.nunique()}")
        print("  voorbeeldwaarden:", reeks.dropna().unique()[:10])


## 3b. TIOS samenvatten per adres

**Wat doet dit blok?**
In TIOS kan één adres meerdere regels hebben. Bijvoorbeeld als iemand eerst dakisolatie en later spouwmuurisolatie liet doen. Voor het model willen we per adres **één regel**. Dit blok vat daarom alle regels van hetzelfde adres (`vbo_id`) samen:

- hoeveel TIOS-regels het adres heeft;
- hoeveel offertes er zijn gemaakt;
- in welk jaar de laatste order was;
- hoeveel verschillende werkzaamheden er waren;
- en het belangrijkste: **heeft dit adres ooit een order-status gehad?** (`heeft_order_status`). Dat bepaalt straks of een adres als klant telt.

Hetzelfde "ooit een order?" wordt ook per pand (`pand_id`) berekend. Dat is een terugval voor als de koppeling op adres niet lukt.

De aantallen en het jaartal gebruiken we straks bewust **niet** om te voorspellen. Die zeggen vooral hoe vaak Takkenkamp al contact had met het adres, en dat ligt te dicht bij het antwoord zelf.


In [ ]:
tios["_jaar"] = pd.to_datetime(tios[TIOS_KOLOM_DATUM], errors="coerce").dt.year
tios["_is_order"] = tios[TIOS_KOLOM_STATUS].isin(ORDER_STATUSSEN)

_t0 = time.time()
tios_agg = tios.groupby("vbo_id").agg(
    aantal_tios_regels=(TIOS_KOLOM_ID, "count"),
    aantal_offertes=(TIOS_KOLOM_OFFERTES, "sum"),
    laatste_order_jaar=("_jaar", "max"),
    heeft_order_status=("_is_order", "any"),
    unieke_werkzaamheden=(TIOS_KOLOM_SOORT, "nunique"),
).reset_index()

tios_pand_status = (
    tios.groupby("pand_id")["_is_order"].any()
    .reset_index()
    .rename(columns={"_is_order": "heeft_order_status_pand"})
)
print(f"TIOS samengevat per adres in {time.time() - _t0:.1f} seconden ({len(tios_agg)} unieke adressen).")


## 3c. Gemiddeld gasverbruik per buurt toevoegen (CBS)

**Wat doet dit blok?**
Dit blok haalt bij het CBS het gemiddelde gasverbruik per buurt op en zet dat bij elk adres. Het idee: een buurt waar veel gas wordt gestookt, is waarschijnlijk slechter geïsoleerd, en dus kansrijker voor Takkenkamp.

Stap voor stap:
1. De functie `cbs_odata_ophalen` haalt de cijfers op van de CBS-website. Het CBS geeft de data in pagina's, dus de functie blijft ophalen tot alle pagina's binnen zijn.
2. We halen twee dingen op: de cijfers zelf (per buurtcode) en een lijst die zegt welke naam bij welke code hoort.
3. De functie `normaliseer_naam` maakt buurt- en gemeentenamen gelijk van vorm (kleine letters, geen extra spaties). Anders zou "Zierikzee " niet matchen met "zierikzee".
4. Eerst koppelen we op **buurt + gemeente**. Lukt dat niet, dan vullen we het **gemeentegemiddelde** in.

**Als het misgaat:** heeft je omgeving geen toegang tot de CBS-website, dan stopt het notebook niet. Het meldt de fout en gaat verder zonder gasverbruik (de kolom blijft dan leeg). In Google Colab werkt het ophalen normaal wel.

Technische details: tabel `86159NED`, meting `M000219` ("Gemiddeld aardgasverbruik"), voor "Totaal woningen" (`T001100`).


In [6]:
GAS_TABEL_ID = "86159NED"
GAS_MEASURE_ID = "M000219"
GAS_WONINGKENMERK_ID = "T001100"
CBS_BASIS_URL = f"https://datasets.cbs.nl/odata/v1/CBS/{GAS_TABEL_ID}"


def cbs_odata_ophalen(url, params=None):
    """Haalt alle pagina's van een CBS OData v4-endpoint op."""
    rijen = []
    while url:
        respons = requests.get(url, params=params, timeout=60)
        respons.raise_for_status()
        pagina = respons.json()
        rijen.extend(pagina["value"])
        url = pagina.get("@odata.nextLink")
        params = None
    return pd.DataFrame(rijen)


def normaliseer_naam(serie):
    schoon = serie.astype(str).str.strip().str.lower()
    schoon = schoon.str.replace(r"\s*\([^)]*\)\s*$", "", regex=True)
    schoon = schoon.str.replace(r"\s+", " ", regex=True)
    return schoon


_t0 = time.time()
try:
    print(f"Cijfers ophalen bij CBS StatLine, tabel {GAS_TABEL_ID} ...")
    observaties = cbs_odata_ophalen(
        f"{CBS_BASIS_URL}/Observations",
        params={"$filter": f"Measure eq \'{GAS_MEASURE_ID}\' and Woningkenmerken eq \'{GAS_WONINGKENMERK_ID}\'"},
    )
    if observaties.empty or "WijkenEnBuurten" not in observaties.columns:
        raise RuntimeError("Geen waarnemingen ontvangen van de CBS API.")

    gasverbruik_per_code = observaties[["WijkenEnBuurten", "Value"]].rename(
        columns={"WijkenEnBuurten": "code", "Value": "gasverbruik"}
    )
    codes = cbs_odata_ophalen(f"{CBS_BASIS_URL}/WijkenEnBuurtenCodes")
    codes = codes[["Identifier", "Title", "DimensionGroupId"]].rename(columns={"Identifier": "code"})
    naam_per_code = codes.set_index("code")["Title"]

    gasverbruik_per_code = gasverbruik_per_code.merge(codes, on="code", how="left")
    gasverbruik_per_code["buurtnaam_cbs"] = gasverbruik_per_code["Title"]
    gemeente_code = gasverbruik_per_code["code"].where(
        gasverbruik_per_code["code"].str.startswith("GM"), gasverbruik_per_code["DimensionGroupId"]
    )
    gasverbruik_per_code["gemeentenaam_cbs"] = gemeente_code.map(naam_per_code)
    gasverbruik_per_code["_buurt_norm"] = normaliseer_naam(gasverbruik_per_code["buurtnaam_cbs"])
    gasverbruik_per_code["_gemeente_norm"] = normaliseer_naam(gasverbruik_per_code["gemeentenaam_cbs"])

    gas_per_buurt = (
        gasverbruik_per_code[gasverbruik_per_code["code"].str.startswith("BU")]
        .dropna(subset=["gasverbruik"])
        .drop_duplicates(["_gemeente_norm", "_buurt_norm"])[["_gemeente_norm", "_buurt_norm", "gasverbruik"]]
    )
    gas_per_gemeente = (
        gasverbruik_per_code[gasverbruik_per_code["code"].str.startswith("GM")]
        .dropna(subset=["gasverbruik"])
        .drop_duplicates("_gemeente_norm")[["_gemeente_norm", "gasverbruik"]]
        .rename(columns={"gasverbruik": "gasverbruik_gemeente"})
    )

    assetmaps["_gemeente_norm"] = normaliseer_naam(assetmaps["gemeente"])
    assetmaps["_buurt_norm"] = normaliseer_naam(assetmaps["buurtnaam"])
    assetmaps = assetmaps.merge(gas_per_buurt, on=["_gemeente_norm", "_buurt_norm"], how="left")
    assetmaps = assetmaps.merge(gas_per_gemeente, on="_gemeente_norm", how="left")
    op_buurtniveau = assetmaps["gasverbruik"].notna().sum()
    assetmaps["gasverbruik"] = assetmaps["gasverbruik"].fillna(assetmaps["gasverbruik_gemeente"])
    assetmaps = assetmaps.drop(columns=["gasverbruik_gemeente", "_gemeente_norm", "_buurt_norm"])

    totaal_gevonden = assetmaps["gasverbruik"].notna().sum()
    print(
        f"Gasverbruik gevonden voor {totaal_gevonden} van {len(assetmaps)} adressen "
        f"(waarvan {op_buurtniveau} op buurtniveau, de rest op gemeenteniveau)."
    )
except Exception as fout:
    print(f"CBS-gasverbruik kon niet worden opgehaald ({fout}).")
    print("Ga door zonder dit kenmerk (het notebook vult dit verderop als ontbrekend in).")
    assetmaps["gasverbruik"] = np.nan

print(f"Stap 3c (gasverbruik) klaar in {time.time() - _t0:.1f} seconden.")


Cijfers ophalen bij CBS StatLine, tabel 86159NED ...


CBS-gasverbruik kon niet worden opgehaald (HTTPSConnectionPool(host='datasets.cbs.nl', port=443): Max retries exceeded with url: /odata/v1/CBS/86159NED/Observations?%24filter=Measure+eq+%27M000219%27+and+Woningkenmerken+eq+%27T001100%27 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))).
Ga door zonder dit kenmerk (het notebook vult dit verderop als ontbrekend in).
Stap 3c (gasverbruik) klaar in 0.2 seconden.


## 3d. Gemiddeld inkomen per buurt toevoegen (CBS)

**Wat doet dit blok?**
Precies hetzelfde als het blok hierboven, maar nu voor het **gemiddelde inkomen per inwoner** in de buurt. Het idee: in buurten met een hoger inkomen hebben mensen misschien meer ruimte om te investeren in verduurzaming.

Ook hier: eerst koppelen op buurt, anders op gemeente. En lukt het ophalen niet, dan gaat het notebook gewoon door zonder dit kenmerk.

Technische details: tabel `85984NED` ("Kerncijfers wijken en buurten"), meting `M000224`.


In [7]:
INKOMEN_TABEL_ID = "85984NED"
INKOMEN_MEASURE_ID = "M000224"
inkomen_basis_url = f"https://datasets.cbs.nl/odata/v1/CBS/{INKOMEN_TABEL_ID}"

_t0 = time.time()
try:
    print(f"Inkomenscijfers ophalen bij CBS StatLine, tabel {INKOMEN_TABEL_ID} ...")
    observaties_inkomen = cbs_odata_ophalen(
        f"{inkomen_basis_url}/Observations",
        params={"$filter": f"Measure eq \'{INKOMEN_MEASURE_ID}\'"},
    )
    if observaties_inkomen.empty or "WijkenEnBuurten" not in observaties_inkomen.columns:
        raise RuntimeError("Geen waarnemingen ontvangen van de CBS API.")

    inkomen_per_code = observaties_inkomen[["WijkenEnBuurten", "Value"]].rename(
        columns={"WijkenEnBuurten": "code", "Value": "inkomen"}
    )
    codes_inkomen = cbs_odata_ophalen(f"{inkomen_basis_url}/WijkenEnBuurtenCodes")
    codes_inkomen = codes_inkomen[["Identifier", "Title", "DimensionGroupId"]].rename(columns={"Identifier": "code"})
    naam_per_code_inkomen = codes_inkomen.set_index("code")["Title"]

    inkomen_per_code = inkomen_per_code.merge(codes_inkomen, on="code", how="left")
    inkomen_per_code["buurtnaam_cbs"] = inkomen_per_code["Title"]
    gemeente_code_inkomen = inkomen_per_code["code"].where(
        inkomen_per_code["code"].str.startswith("GM"), inkomen_per_code["DimensionGroupId"]
    )
    inkomen_per_code["gemeentenaam_cbs"] = gemeente_code_inkomen.map(naam_per_code_inkomen)
    inkomen_per_code["_buurt_norm"] = normaliseer_naam(inkomen_per_code["buurtnaam_cbs"])
    inkomen_per_code["_gemeente_norm"] = normaliseer_naam(inkomen_per_code["gemeentenaam_cbs"])

    inkomen_per_buurt = (
        inkomen_per_code[inkomen_per_code["code"].str.startswith("BU")]
        .dropna(subset=["inkomen"])
        .drop_duplicates(["_gemeente_norm", "_buurt_norm"])[["_gemeente_norm", "_buurt_norm", "inkomen"]]
    )
    inkomen_per_gemeente = (
        inkomen_per_code[inkomen_per_code["code"].str.startswith("GM")]
        .dropna(subset=["inkomen"])
        .drop_duplicates("_gemeente_norm")[["_gemeente_norm", "inkomen"]]
        .rename(columns={"inkomen": "inkomen_gemeente"})
    )

    assetmaps["_gemeente_norm"] = normaliseer_naam(assetmaps["gemeente"])
    assetmaps["_buurt_norm"] = normaliseer_naam(assetmaps["buurtnaam"])
    assetmaps = assetmaps.merge(inkomen_per_buurt, on=["_gemeente_norm", "_buurt_norm"], how="left")
    assetmaps = assetmaps.merge(inkomen_per_gemeente, on="_gemeente_norm", how="left")
    op_buurtniveau_inkomen = assetmaps["inkomen"].notna().sum()
    assetmaps["inkomen"] = assetmaps["inkomen"].fillna(assetmaps["inkomen_gemeente"])
    assetmaps = assetmaps.rename(columns={"inkomen": "gemiddeld_inkomen"})
    assetmaps = assetmaps.drop(columns=["inkomen_gemeente", "_gemeente_norm", "_buurt_norm"])

    totaal_gevonden_inkomen = assetmaps["gemiddeld_inkomen"].notna().sum()
    print(
        f"Inkomen gevonden voor {totaal_gevonden_inkomen} van {len(assetmaps)} adressen "
        f"(waarvan {op_buurtniveau_inkomen} op buurtniveau, de rest op gemeenteniveau)."
    )
except Exception as fout:
    print(f"CBS-inkomen kon niet worden opgehaald ({fout}).")
    print("Ga door zonder dit kenmerk (het notebook vult dit verderop als ontbrekend in).")
    assetmaps["gemiddeld_inkomen"] = np.nan

print(f"Stap 3d (inkomen) klaar in {time.time() - _t0:.1f} seconden.")


Inkomenscijfers ophalen bij CBS StatLine, tabel 85984NED ...


CBS-inkomen kon niet worden opgehaald (HTTPSConnectionPool(host='datasets.cbs.nl', port=443): Max retries exceeded with url: /odata/v1/CBS/85984NED/Observations?%24filter=Measure+eq+%27M000224%27 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))).
Ga door zonder dit kenmerk (het notebook vult dit verderop als ontbrekend in).
Stap 3d (inkomen) klaar in 0.2 seconden.


## 4. TIOS en Assetmaps koppelen, en bepalen wie klant is

**Wat doet dit blok?**
Assetmaps bevat (bijna) alle adressen in het gebied. TIOS bevat alleen adressen waar Takkenkamp mee te maken had. Dit blok legt de twee naast elkaar: bij elk Assetmaps-adres zoeken we de TIOS-samenvatting uit stap 3b.

- Eerst koppelen op `vbo_id` (het adres).
- Daarnaast koppelen op `pand_id` (het gebouw), als terugval.

Daarna maken we de kolom `is_klant`. Dit is wat het model moet leren voorspellen (de **doelvariabele**). Een adres is klant (`1`) als minstens één van deze drie klopt:
1. Assetmaps zegt zelf dat het klant is (kolom `klant`);
2. het adres heeft in TIOS een order-status gehad;
3. het pand heeft in TIOS een order-status gehad.

Anders is het `0` (geen klant).

Het blok print ook hoeveel adressen **niet** aan TIOS gekoppeld zijn. Dat zijn er veel, en dat is normaal: de meeste adressen zijn nooit met Takkenkamp in contact geweest. Tot slot zie je hoeveel procent klant is. Dat is erg weinig (minder dan 1%), en daar houdt het model rekening mee.


In [8]:
koppeling = assetmaps.merge(tios_agg, on="vbo_id", how="left")
koppeling = koppeling.merge(tios_pand_status, on="pand_id", how="left")

niet_gematcht = koppeling["aantal_tios_regels"].isna().sum()
print(f"Niet gekoppeld aan een TIOS-record: {niet_gematcht} van {len(koppeling)} adressen")
print("(dat is normaal — niet elk Assetmaps-adres heeft ooit een lead/order gehad)")

is_al_klant_vlag = koppeling["klant"].astype(str).str.lower().isin(["1", "true", "ja", "j"])
koppeling["is_klant"] = (
    is_al_klant_vlag
    | koppeling["heeft_order_status"].fillna(False)
    | koppeling["heeft_order_status_pand"].fillna(False)
).astype(int)

print("\nVerdeling doelvariabele (1 = klant):")
print(koppeling["is_klant"].value_counts())
print(f"Aandeel klant: {koppeling['is_klant'].mean()*100:.2f}%")


Niet gekoppeld aan een TIOS-record: 83620 van 84031 adressen
(dat is normaal — niet elk Assetmaps-adres heeft ooit een lead/order gehad)

Verdeling doelvariabele (1 = klant):
is_klant
0    83620
1      411
Name: count, dtype: int64
Aandeel klant: 0.49%


## 5. Ontbrekende waarden opschonen

**Wat doet dit blok?**
Twee dingen:

1. **-99997 vervangen door "leeg".** Assetmaps schrijft -99997 als het een waarde niet weet (bijvoorbeeld de WOZ-waarde). Als we dat laten staan, denkt het model dat dit een echt (heel laag) getal is. Daarom maken we er een lege waarde van.
2. **Een "ontbreekt"-kolom toevoegen.** Voor elke kolom waar meer dan 0,5% van de waarden leeg is, maken we een extra kolom met 1 (leeg) of 0 (ingevuld). Bijvoorbeeld `woz_ontbreekt`. Waarom? Soms zegt het ontbreken van een waarde zelf al iets. Straks vullen we de lege plekken op met een gemiddelde waarde, en zonder deze extra kolom zou die informatie verloren gaan.

Aan het eind zie je voor welke kolommen zo'n extra "ontbreekt"-kolom is gemaakt.


In [ ]:
for kolom in ["huur", "koop", "woz"]:
    if kolom in koppeling.columns:
        koppeling[kolom] = koppeling[kolom].replace(MISSING_CODE, np.nan)

# "huur" bewust weggelaten: we gebruiken alleen "koop" (zie stap 8), dus een
# missing-indicator voor "huur" heeft geen nut.
KOLOMMEN_MET_MOGELIJK_MISSENDE_WAARDE = [
    "vbo_oppervlakte", "pandbouwjaar", "pandoppervlakte", "koop", "woz", "gasverbruik", "gemiddeld_inkomen",
]
MISSING_INDICATOR_DREMPEL = 0.005  # 0,5% van de adressen

MISSING_INDICATOR_KENMERKEN = []
for kolom in KOLOMMEN_MET_MOGELIJK_MISSENDE_WAARDE:
    if kolom not in koppeling.columns:
        continue
    aandeel_missend = koppeling[kolom].isna().mean()
    if aandeel_missend > MISSING_INDICATOR_DREMPEL:
        indicator_naam = f"{kolom}_ontbreekt"
        koppeling[indicator_naam] = koppeling[kolom].isna().astype(int)
        MISSING_INDICATOR_KENMERKEN.append(indicator_naam)

print("Missing-indicatoren aangemaakt voor:", MISSING_INDICATOR_KENMERKEN)


## 6. Nieuwe kenmerken maken

**Wat doet dit blok?**
Het model wordt beter als we het slimme, bruikbare kenmerken geven. Dit blok maakt er een paar bij op basis van wat we al hebben:

- `pand_leeftijd`: hoe oud het pand is (huidig jaar min bouwjaar).
- `voor_spouwmuurnorm_1975`: 1 als het pand vóór 1975 is gebouwd. Toen werd spouwmuurisolatie pas gangbaar, dus oudere huizen hebben dat vaak nog niet.
- `voor_hrpp_norm_1995`: 1 als het pand vóór 1995 is gebouwd, de tijd van de HR++-glasnorm.
- `energieklasse_rang`: het energielabel als getal (A=1 ... G=7). Zo snapt het model dat een G-label "slechter" is dan een C-label.
- `buurtnaam_frequentie`: hoeveel adressen er in de buurt liggen. Zo zegt de buurt iets over grootte, zonder dat we honderden buurtnamen als losse kolommen nodig hebben.
- `postcode4`: de eerste 4 cijfers van de postcode, nodig voor het buurteffect in stap 7.

Het blok print hoeveel unieke buurten, postcodegebieden en gemeenten er in de data zitten.


In [10]:
koppeling["pand_leeftijd"] = HUIDIG_JAAR - koppeling["pandbouwjaar"]
koppeling["voor_spouwmuurnorm_1975"] = (koppeling["pandbouwjaar"] < 1975).astype(int)
koppeling["voor_hrpp_norm_1995"] = (koppeling["pandbouwjaar"] < 1995).astype(int)

ENERGIE_RANG = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
koppeling["energieklasse_rang"] = koppeling["energieklasse"].map(ENERGIE_RANG)

buurt_frequenties = koppeling["buurtnaam"].value_counts()
koppeling["buurtnaam_frequentie"] = koppeling["buurtnaam"].map(buurt_frequenties)
koppeling["postcode4"] = koppeling["postcode"].astype(str).str[:4]

print(f"{koppeling['buurtnaam'].nunique()} unieke buurten, {koppeling['postcode4'].nunique()} unieke postcode4-gebieden, {koppeling['gemeente'].nunique()} gemeentes.")


423 unieke buurten, 152 unieke postcode4-gebieden, 13 gemeentes.


## 7. Het buurteffect: bestaande klanten in de buurt

**Wat doet dit blok?**
Isolatiewerk verspreidt zich vaak via de buurt: buren zien de steiger staan en raken erover in gesprek. Daarom tellen we per adres hoeveel adressen in dezelfde **buurt** en hetzelfde **postcode4-gebied** al klant zijn.

Het lastige: het model mag bij het tellen nooit (stiekem) het antwoord zien. Daarvoor zijn er twee regels.

**Regel 1: adressen waarop het model leert (trainingsdata).**
We knippen de trainingsdata in 5 stukken. Voor de adressen in stuk 1 tellen we de klanten in de andere 4 stukken. Voor stuk 2 tellen we de klanten in stuk 1, 3, 4 en 5. Enzovoort. Een adres telt zo nooit zichzelf mee.

*Waarom niet gewoon "totaal min het adres zelf"?* Dat deed de vorige versie. Dan krijgt een klant altijd precies 1 minder dan zijn buren die geen klant zijn. Het model leert dan het trucje "1 lager = klant", en dat trucje werkt niet bij nieuwe adressen. Bij `gemeente_klanten_nabij` ging dit zo mis dat het model er slechter van werd. Die is daarom helemaal verwijderd (`gemeente` zelf zit er nog wel in).

Omdat we steeds maar 4 van de 5 stukken tellen, komt de telling ongeveer 20% te laag uit. Daarom vermenigvuldigen we hem met 5/4. Zo is hij even groot als de telling bij regel 2.

**Regel 2: adressen die we willen beoordelen (testdata of de leadlijst).**
Hier tellen we gewoon alle klanten in de trainingsdata. Het adres zelf zit daar niet in, dus spieken kan niet.

De functies:
- `regio_kenmerken_train` – regel 1.
- `regio_kenmerken_score` – regel 2.
- `maak_regio_kenmerken_out_of_fold` – combineert beide voor het testen in stap 10 en 11.


**Waarom log1p?** Zonder demping zou een klein aantal buurten met heel veel bestaande klanten alle andere kenmerken overstemmen, en zou het model bijna alleen nog maar "buren van klanten" bovenaan zetten. log1p behoudt de volgorde maar maakt het verschil tussen bijvoorbeeld 20 en 25 buren-die-al-klant-zijn veel kleiner dan tussen 0 en 5. Zie stap 10b en 12b voor de gebieden waar dit kenmerk sowieso 0 is.

In [ ]:
# Welke groepen we gebruiken voor het buurteffect (gemeente is bewust weggelaten)
REGIO_KOLOMMEN = [
    ("buurtnaam", "buurt_klanten_nabij"),
    ("postcode4", "postcode4_klanten_nabij"),
]
BINNEN_FOLDS = 5  # in hoeveel stukken de trainingsdata wordt geknipt (regel 1)


def _dempen(resultaat):
    """Dempt de ruwe tellingen met log1p. Zonder dit trekt een klein aantal
    buurten met heel veel bestaande klanten (bijv. 20+) de rest van de
    kenmerken volledig weg: het verschil tussen 0 en 3 klanten in de buurt
    is voor het model dan veel belangrijker dan bijvoorbeeld bouwjaar of
    energielabel. log1p behoudt de volgorde (meer buren = hogere waarde,
    0 blijft 0) maar maakt het verschil tussen bijv. 20 en 25 klanten veel
    kleiner dan tussen 0 en 5. Zie ook stap 10b en 12b hieronder."""
    return {naam: np.log1p(serie) for naam, serie in resultaat.items()}


def regio_kenmerken_train(train):
    """Regel 1: buurteffect voor de trainingsdata. Elk adres krijgt het aantal klanten
    uit de ándere stukken van de trainingsdata, zodat het zichzelf nooit meetelt."""
    resultaat = {naam: pd.Series(0.0, index=train.index) for _, naam in REGIO_KOLOMMEN}
    binnen_cv = KFold(n_splits=BINNEN_FOLDS, shuffle=True, random_state=42)
    ophoging = BINNEN_FOLDS / (BINNEN_FOLDS - 1)  # corrigeert voor het missende stuk

    for tel_pos, doel_pos in binnen_cv.split(train):
        tel_deel = train.iloc[tel_pos]
        doel_deel = train.iloc[doel_pos]
        for kolom, naam in REGIO_KOLOMMEN:
            telling = tel_deel.groupby(kolom)["is_klant"].sum()
            resultaat[naam].loc[doel_deel.index] = (
                doel_deel[kolom].map(telling).fillna(0).values * ophoging
            )
    return _dempen(resultaat)


def regio_kenmerken_score(train, nieuw_deel):
    """Regel 2: buurteffect voor adressen die we beoordelen. Telt alle klanten
    in de trainingsdata."""
    resultaat = {}
    for kolom, naam in REGIO_KOLOMMEN:
        telling = train.groupby(kolom)["is_klant"].sum()
        resultaat[naam] = nieuw_deel[kolom].map(telling).fillna(0).astype(float)
    return _dempen(resultaat)


def maak_regio_kenmerken_out_of_fold(data, train_idx, test_idx):
    """Voor het testen (stap 10 en 11): regel 1 voor de trainrijen, regel 2 voor de testrijen."""
    train = data.loc[train_idx]
    test = data.loc[test_idx]
    deel_train = regio_kenmerken_train(train)
    deel_test = regio_kenmerken_score(train, test)
    return {naam: pd.concat([deel_train[naam], deel_test[naam]]).reindex(data.index)
            for _, naam in REGIO_KOLOMMEN}


print("Hulpfuncties voor het buurteffect klaar (buurt en postcode4, gedempt met log1p).")


## 8. Kenmerken kiezen

**Wat doet dit blok?**
Hier bepalen we welke kolommen het model mag gebruiken om te voorspellen. Dat zijn de **kenmerken** (features). Er zijn twee soorten:

- **Numerieke kenmerken** (getallen): oppervlakte, bouwjaar, leeftijd, WOZ, huur/koop, gasverbruik, inkomen, de nieuwe kenmerken uit stap 6, het buurteffect uit stap 7 (buurt en postcode4) en de "ontbreekt"-kolommen uit stap 5.
- **Categorische kenmerken** (tekst/groepen): alleen `gemeente`.

Bewust **niet** gebruikt: gegevens over de order zelf, zoals `status`, `offertes` of het aantal TIOS-regels. Die ontstaan pas als iemand al met Takkenkamp in gesprek is. Het model zou dan het antwoord "voorzeggen" in plaats van echt voorspellen. `provincie` en `gebruiksfunctie` zijn ook weggelaten, omdat ze in een eerdere test niets toevoegden.

Tot slot maken we `data` (de tabel waarmee we werken) en `y` (de kolom `is_klant`: het antwoord dat het model moet leren).


In [ ]:
# "huur" bewust niet meegenomen: Takkenkamp richt zich op eigenaar-bewoners
# (huurders beslissen niet zelf over verduurzaming), dus alleen "koop" gebruiken.
NUMERIEKE_KENMERKEN = [
    "vbo_oppervlakte", "pandbouwjaar", "pand_leeftijd", "pandoppervlakte", "koop", "woz",
    "gasverbruik", "gemiddeld_inkomen", "buurtnaam_frequentie",
    "voor_spouwmuurnorm_1975", "voor_hrpp_norm_1995", "energieklasse_rang",
    "buurt_klanten_nabij", "postcode4_klanten_nabij",
] + MISSING_INDICATOR_KENMERKEN
CATEGORISCHE_KENMERKEN = ["gemeente"]

kenmerken = NUMERIEKE_KENMERKEN + CATEGORISCHE_KENMERKEN
data = koppeling.dropna(subset=["is_klant"]).reset_index(drop=True)
y = data["is_klant"]

print(f"Dataset voor modellering: {len(data)} rijen, {y.sum()} positief ({y.mean()*100:.2f}%)")


## 9. Voorbewerking: data geschikt maken voor het model

**Wat doet dit blok?**
Een model kan niet omgaan met lege plekken of met tekst. Dit blok maakt een "voorbewerkingsstap" die dat oplost. Die stap zit straks automatisch vóór elk model:

- **Getallen:** lege plekken worden gevuld met de **mediaan** (de middelste waarde) van die kolom.
- **Gemeente (tekst):** lege plekken worden "onbekend". Daarna wordt elke gemeente een eigen 0/1-kolom (*one-hot*). Bijvoorbeeld: `gemeente_Veere` = 1 als het adres in Veere ligt, anders 0.

Hier wordt nog niets berekend, alleen het recept klaargezet.


In [13]:
voorbewerking = ColumnTransformer([
    ("numeriek", SimpleImputer(strategy="median"), NUMERIEKE_KENMERKEN),
    ("categorisch", Pipeline([
        ("opvullen", SimpleImputer(strategy="constant", fill_value="onbekend")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORISCHE_KENMERKEN),
])
print("Preprocessing-pipeline klaar.")


Preprocessing-pipeline klaar.


## 10. Het model testen met kruisvalidatie (3 folds)

**Wat doet dit blok?**
Voordat we het model op alle adressen loslaten, willen we weten hoe goed het is. Dat doen we met **kruisvalidatie**:

1. We verdelen alle adressen in **3 gelijke stukken** (folds). Elk stuk heeft ongeveer evenveel klanten.
2. We trainen het model op 2 stukken en testen op het 3e stuk. Het model heeft dat 3e stuk dus nooit gezien.
3. Dat doen we 3 keer, zodat elk stuk één keer het teststuk is.
4. We nemen het gemiddelde van de 3 scores.

Per fold wordt het buurteffect (stap 7) opnieuw uitgerekend met alleen de trainingsdata, zodat het model niet kan spieken.

**Het model.** De functie `maak_ensemble` maakt de twee modellen:
- *Logistic regression*: een eenvoudig model dat per kenmerk een gewicht leert (bijvoorbeeld "ouder pand = iets kansrijker").
- *RandomForest*: een model dat heel veel beslisbomen maakt ("is het pand van vóór 1975? zo ja, is de oppervlakte groter dan 120 m²? ...") en die laat stemmen.

`voorspel_ensemble` neemt het gemiddelde van de twee kansen. Beide modellen krijgen een extra gewicht op klanten (`class_weight="balanced"`), omdat er zo weinig klanten zijn. Anders zou het model leren om gewoon altijd "geen klant" te zeggen.

**Wat meten we?**
- **AUC**: hoe goed het model klanten boven niet-klanten zet in de ranglijst. 0,5 = gokken, 1,0 = perfect. Met zo weinig klanten is 0,65–0,70 een eerlijk en bruikbaar resultaat. Zie je boven de 0,9? Wantrouw dat en controleer eerst op spieken.
- **Precision@k**: van de top-k adressen die het model aanwijst (bijvoorbeeld de top-100), welk percentage is echt klant? Dit is de belangrijkste maat voor het salesteam, want die belt of schrijft alleen de bovenkant van de lijst aan. Vergelijk het met de "toevalskans": het percentage dat je zou halen door willekeurig adressen te kiezen.

Dit is het zwaarste blok van het notebook, maar doordat er nu maar 3 folds en 1 model (ensemble) zijn, is het een stuk sneller dan voorheen.


In [ ]:
AANTAL_FOLDS = 3
cv = StratifiedKFold(n_splits=AANTAL_FOLDS, shuffle=True, random_state=42)


def maak_ensemble():
    """Maakt de twee modellen van het ensemble: logistic regression + RandomForest.
    Dit is hetzelfde model dat in stap 12 de leadlijst maakt. clone() geeft elk
    model zijn eigen kopie van de voorbewerking, zodat ze elkaar niet beïnvloeden."""
    log_model = Pipeline([
        ("prep", clone(voorbewerking)),
        ("clf", LogisticRegression(max_iter=500, C=1.0, class_weight="balanced")),
    ])
    rf_model = Pipeline([
        ("prep", clone(voorbewerking)),
        ("clf", RandomForestClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=30,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1,
        )),
    ])
    return log_model, rf_model


def voorspel_ensemble(log_model, rf_model, X):
    """Gemiddelde van de kans volgens logistic regression en volgens RandomForest."""
    return (log_model.predict_proba(X)[:, 1] + rf_model.predict_proba(X)[:, 1]) / 2


def precision_at_k(y_werkelijk, kans, k):
    """Aandeel echte klanten in de k adressen met de hoogste kans."""
    volgorde = np.argsort(-kans)[:k]
    return y_werkelijk.iloc[volgorde].mean()


K_WAARDEN = [50, 100, 250, 500]
auc_scores = []
precision_resultaten = {k: [] for k in K_WAARDEN}

_t0 = time.time()
for fold, (train_idx, test_idx) in enumerate(cv.split(data, y)):
    # Buurteffect opnieuw berekenen met alleen de trainingsdata van deze fold
    regio_kenmerken = maak_regio_kenmerken_out_of_fold(data, train_idx, test_idx)
    fold_data = data.copy()
    for naam_kenmerk, serie in regio_kenmerken.items():
        fold_data[naam_kenmerk] = serie

    X_train, X_test = fold_data.loc[train_idx, kenmerken], fold_data.loc[test_idx, kenmerken]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    # Ensemble trainen en de test-adressen scoren (één keer per fold)
    log_model, rf_model = maak_ensemble()
    log_model.fit(X_train, y_train)
    rf_model.fit(X_train, y_train)
    kans = voorspel_ensemble(log_model, rf_model, X_test)

    # Scores bijhouden
    auc_scores.append(roc_auc_score(y_test, kans))
    for k in K_WAARDEN:
        if k <= len(X_test):
            precision_resultaten[k].append(precision_at_k(y_test.reset_index(drop=True), kans, k))

    print(f"Fold {fold + 1}/{AANTAL_FOLDS} klaar ({time.time() - _t0:.1f}s totaal)")

print(f"\nAUC ensemble (logistic + RF): {np.mean(auc_scores):.3f} (+/- {np.std(auc_scores):.3f})")

print(f"\nPrecision@k (aandeel echte klanten in de top-k van elke test-fold, gemiddeld over {AANTAL_FOLDS} folds):")
for k, scores in precision_resultaten.items():
    if scores:
        print(f"  top-{k:4d}: {np.mean(scores)*100:5.1f}%  (toevalskans: {y.mean()*100:.2f}%)")


## 10b. Extra check: hoe goed werkt dit in postcodegebieden die het model nog nooit zag?

**Waarom dit blok?**
De kruisvalidatie hierboven (stap 10) knipt de adressen willekeurig in stukken. Daardoor zit een postcodegebied vaak deels in de trainingsdata én deels in de testdata: het buurteffect (stap 7) kent dat gebied dan al een beetje, ook al is het specifieke adres zelf nieuw voor het model. Dat geeft een te optimistisch beeld voor gebieden waar Takkenkamp nog helemaal nooit is geweest.

Dit blok herhaalt dezelfde test, maar knipt nu op **postcode4-niveau** met `GroupKFold`: een heel postcodegebied zit óf volledig in de training, óf volledig in de test — nooit allebei. Voor de testadressen is het buurteffect dan écht 0, precies zoals bij een gloednieuw gebied. Dit getal is dus een eerlijkere inschatting van "hoe goed vinden we kansrijke adressen in postcodegebieden waar we nog nooit zijn geweest" — en het verschil met de AUC/precision@k uit stap 10 laat zien hoe zwaar het model op het buurteffect leunt versus op de pandkenmerken zelf.

*Kanttekening: een enkele buurt kan in zeldzame gevallen over twee postcode4-gebieden lopen, dus dit is een goede benadering van een "nieuw gebied", geen waterdichte garantie.*


In [ ]:
AANTAL_GROEP_FOLDS = 5
groep_cv = GroupKFold(n_splits=AANTAL_GROEP_FOLDS)
groepen = data["postcode4"]

auc_scores_koud = []
precision_resultaten_koud = {k: [] for k in K_WAARDEN}

_t0 = time.time()
for fold, (train_idx, test_idx) in enumerate(groep_cv.split(data, y, groups=groepen)):
    # Buurteffect opnieuw berekenen, nu met hele postcodegebieden buitengesloten
    regio_kenmerken = maak_regio_kenmerken_out_of_fold(data, train_idx, test_idx)
    fold_data = data.copy()
    for naam_kenmerk, serie in regio_kenmerken.items():
        fold_data[naam_kenmerk] = serie

    X_train, X_test = fold_data.loc[train_idx, kenmerken], fold_data.loc[test_idx, kenmerken]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    log_model, rf_model = maak_ensemble()
    log_model.fit(X_train, y_train)
    rf_model.fit(X_train, y_train)
    kans = voorspel_ensemble(log_model, rf_model, X_test)

    auc_scores_koud.append(roc_auc_score(y_test, kans))
    for k in K_WAARDEN:
        if k <= len(X_test):
            precision_resultaten_koud[k].append(precision_at_k(y_test.reset_index(drop=True), kans, k))

    print(f"Postcode-fold {fold + 1}/{AANTAL_GROEP_FOLDS} klaar ({time.time() - _t0:.1f}s totaal)")

print(f"\nAUC in nooit eerder geziene postcodegebieden: {np.mean(auc_scores_koud):.3f} (+/- {np.std(auc_scores_koud):.3f})")
print("Vergelijk dit met de AUC uit stap 10 — een groter gat betekent dat het model "
      "sterker op het buurteffect leunt dan op de pandkenmerken zelf.")

print("\nPrecision@k in nooit eerder geziene postcodegebieden:")
for k, scores in precision_resultaten_koud.items():
    if scores:
        print(f"  top-{k:4d}: {np.mean(scores)*100:5.1f}%  (toevalskans: {y.mean()*100:.2f}%)")


## 11. Wat drijft het model? Belangrijkste kenmerken

**Wat doet dit blok?**
Dit blok laat zien **welke kenmerken het zwaarst wegen** bij de voorspelling. Handig om aan je stagebegeleider en aan Takkenkamp uit te leggen wáárom een adres kansrijk is.

Zo werkt het:
1. We splitsen de data één keer in 80% training en 20% test.
2. We trainen het ensemble (dezelfde `maak_ensemble` als in stap 10) op de 80% en printen de AUC op de 20%. Dat is een extra controle naast de kruisvalidatie.
3. Dan doen we de **permutatietest** op het RandomForest-deel: we husselen één kolom (bijvoorbeeld `pandbouwjaar`) willekeurig door elkaar en kijken hoeveel slechter het model wordt. Wordt het veel slechter, dan was die kolom belangrijk. Verandert er niets, dan deed die kolom er niet toe. Dat herhalen we voor elke kolom.

Om tijd te sparen gebruiken we hiervoor maximaal 20.000 test-adressen.

**Hoe lees je de uitkomst?** Bovenaan staan de belangrijkste kenmerken. Een getal rond 0 betekent: dit kenmerk helpt nauwelijks. Een negatief getal betekent dat het kenmerk op deze testset eerder ruis gaf dan hulp.


In [ ]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    data, y, test_size=0.2, random_state=42, stratify=y
)

# Buurteffect alleen op basis van de trainingsdata (geen spieken)
regio_train = maak_regio_kenmerken_out_of_fold(data, X_train_full.index, X_test_full.index)
data_met_regio = data.copy()
for naam_kenmerk, serie in regio_train.items():
    data_met_regio[naam_kenmerk] = serie

# Hetzelfde ensemble als in stap 10
eind_model_log, eind_model_rf = maak_ensemble()

eind_model_rf.fit(data_met_regio.loc[X_train_full.index, kenmerken], y_train_full)
eind_model_log.fit(data_met_regio.loc[X_train_full.index, kenmerken], y_train_full)

X_test_kenmerken = data_met_regio.loc[X_test_full.index, kenmerken]
ensemble_test_kans = voorspel_ensemble(eind_model_log, eind_model_rf, X_test_kenmerken)
print("AUC van het ensemble op de achtergehouden testset:", round(roc_auc_score(y_test_full, ensemble_test_kans), 3))

# Permutatietest op het RandomForest-deel
_t0 = time.time()
PERMUTATIE_STEEKPROEF = 20_000
if len(X_test_kenmerken) > PERMUTATIE_STEEKPROEF:
    steekproef_idx = X_test_kenmerken.sample(PERMUTATIE_STEEKPROEF, random_state=42).index
else:
    steekproef_idx = X_test_kenmerken.index

resultaat_permutatie = permutation_importance(
    eind_model_rf, X_test_kenmerken.loc[steekproef_idx], y_test_full.loc[steekproef_idx],
    scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=1,
)
print(f"Feature-belangrijkheid berekend in {time.time() - _t0:.1f} seconden ({len(steekproef_idx)} rijen).")

belangrijkheid = pd.Series(
    resultaat_permutatie.importances_mean, index=X_test_kenmerken.columns,
).sort_values(ascending=False)

print("\nBelangrijkste kenmerken (afname in AUC als je de kolom door elkaar husselt):")
print(belangrijkheid)


## 12. De leadlijst maken

**Wat doet dit blok?**
Dit is het eindresultaat. Het blok:

1. **Traint het ensemble opnieuw, nu op álle adressen.** Hoe meer voorbeelden, hoe beter het model leert. Het is precies hetzelfde model als in stap 10 (`maak_ensemble`). Het buurteffect voor deze trainingsdata wordt berekend met regel 1 uit stap 7.
2. **Berekent het buurteffect voor de adressen die nog géén klant zijn**, met regel 2: alle bekende klanten tellen mee.
3. **Scoort die adressen.** Elk adres krijgt een `kans_op_klant` tussen 0 en 1. Hoe hoger, hoe kansrijker.
4. **Sorteert van hoog naar laag** en houdt alleen de handige kolommen over: adres, postcode, woonplaats, bouwjaar, gemeente, provincie en de kans.
5. **Schrijft alles naar `leadgenerator_resultaat.xlsx`.** Dit bestand kun je openen in Excel en filteren, bijvoorbeeld op gemeente voor een brievenactie.

Onderaan zie je de top-20 kansrijkste adressen.

**Let op bij het lezen van de kans:** het getal is vooral bedoeld om adressen te **rangschikken**. Omdat het model extra gewicht op klanten legt, is een kans van bijvoorbeeld 0,6 niet letterlijk "60% kans dat dit adres klant wordt". Kijk dus vooral naar de volgorde.

Duurt het wegschrijven naar Excel te lang? Dan kun je de regel met `.to_csv` gebruiken (staat als opmerking in de code). Dat is veel sneller.


In [ ]:
_t0 = time.time()

# 1. Ensemble trainen op alle adressen (buurteffect volgens regel 1)
train_volledig = koppeling.loc[data.index].copy()
for naam_kenmerk, serie in regio_kenmerken_train(train_volledig).items():
    train_volledig[naam_kenmerk] = serie

eind_model_log_volledig, eind_model_rf_volledig = maak_ensemble()
eind_model_log_volledig.fit(train_volledig[kenmerken], y)
eind_model_rf_volledig.fit(train_volledig[kenmerken], y)

# 2. Adressen die nog geen klant zijn: buurteffect volgens regel 2 (alle klanten tellen)
nog_geen_klant = koppeling[koppeling["is_klant"] == 0].copy()
for naam_kenmerk, serie in regio_kenmerken_score(train_volledig, nog_geen_klant).items():
    nog_geen_klant[naam_kenmerk] = serie
nog_geen_klant = nog_geen_klant.dropna(subset=kenmerken, how="all")

# 3. Scoren
nog_geen_klant["kans_op_klant"] = voorspel_ensemble(
    eind_model_log_volledig, eind_model_rf_volledig, nog_geen_klant[kenmerken]
)
print(f"Voorspelling voor {len(nog_geen_klant)} adressen klaar in {time.time() - _t0:.1f} seconden.")

# 4. Lijst sorteren en opslaan
output_kolommen = [
    "straatnaam", "huisnummer", "postcode", "woonplaats",
    "pandbouwjaar", "gemeente", "provincie", "kans_op_klant",
]
resultaat = (
    nog_geen_klant[output_kolommen]
    .sort_values("kans_op_klant", ascending=False)
    .rename(columns={"pandbouwjaar": "bouwjaar"})
)

_t0 = time.time()
resultaat.to_excel("leadgenerator_resultaat.xlsx", index=False)
print(f"{len(resultaat)} adressen weggeschreven naar leadgenerator_resultaat.xlsx in {time.time() - _t0:.1f} seconden.")
# Is dit wegschrijven zelf traag? resultaat.to_csv("leadgenerator_resultaat.csv", index=False)
# is dan een veel sneller alternatief (met dezelfde inhoud, wel .csv i.p.v. .xlsx).
resultaat.head(20)


## 12b. Aparte lijst: adressen in postcodegebieden zonder bestaande klant

**Wat doet dit blok?**
De hoofdlijst uit stap 12 zet vaak buren van bestaande klanten bovenaan — logisch, want "isolatiewerk verspreidt zich via de buurt" is een sterk patroon in de data. Fijn voor cross-sell in bekende straten, maar niet voor het opsporen van kansrijke nieuwe gebieden.

Dit blok filtert de leadlijst tot adressen waar het buurteffect **nul** is: geen bekende klant in dezelfde buurt én in hetzelfde postcode4-gebied. De score van die adressen komt dan vrijwel volledig uit de pandkenmerken zelf (bouwjaar, energielabel, oppervlakte, gasverbruik, aandeel koopwoningen) — het "klantprofiel" van het model, losgetrokken van waar Takkenkamp toevallig al is geweest. Handig om nieuwe postcodegebieden of regio's te selecteren voor een gerichte actie.

**Let op bij het lezen:** in deze gebieden zijn er per definitie geen historische klanten om tegen te toetsen, dus precision@k is hier niet rechtstreeks te meten. Gebruik de score als een gemotiveerde eerste selectie, niet als harde garantie — stap 10b hierboven geeft wel een eerlijke schatting van hoe goed dit in de praktijk werkt.


In [ ]:
witte_vlekken = nog_geen_klant[
    (nog_geen_klant["buurt_klanten_nabij"] == 0) & (nog_geen_klant["postcode4_klanten_nabij"] == 0)
].copy()

resultaat_witte_vlekken = (
    witte_vlekken[output_kolommen]
    .sort_values("kans_op_klant", ascending=False)
    .rename(columns={"pandbouwjaar": "bouwjaar"})
)

print(
    f"{len(resultaat_witte_vlekken)} van de {len(resultaat)} adressen liggen in een "
    "postcodegebied zonder bekende klant."
)

resultaat_witte_vlekken.to_excel("leadgenerator_witte_vlekken.xlsx", index=False)
print("Weggeschreven naar leadgenerator_witte_vlekken.xlsx.")
# Ook hier geldt: .to_csv("leadgenerator_witte_vlekken.csv", index=False) is sneller
# als wegschrijven naar Excel te lang duurt.
resultaat_witte_vlekken.head(20)
